# Quickstart — AgentCore SDK (`MemoryClient`)

End-to-end walkthrough of AgentCore Memory using `bedrock_agentcore.memory.MemoryClient` — the higher-level Python client that wraps the raw boto3 APIs with helpers like `create_memory_and_wait`, `get_last_k_turns`, and `retrieve_memories`.

The [CLI](./03-quickstart-cli.md) and [boto3](./04-quickstart-boto3.ipynb) quickstarts cover the same flow with different surfaces.

## Prerequisites

- AWS credentials for a region where AgentCore Memory is available.
- An IAM **memory execution role ARN** for long-term extraction.
- Amazon Bedrock access to the embedding model used by semantic strategies.
- `pip install bedrock-agentcore`

In [ ]:
import os
import time

from bedrock_agentcore.memory import MemoryClient

REGION = os.getenv("AWS_REGION", "us-east-1")
MEMORY_ROLE_ARN = os.environ["MEMORY_EXECUTION_ROLE_ARN"]
ACTOR_ID = "user-42"
SESSION_ID = f"sess-{int(time.time())}"

client = MemoryClient(region_name=REGION)

## 1. Create a memory resource

`create_memory_and_wait` creates the resource and blocks until it reaches `ACTIVE`.

In [ ]:
memory = client.create_memory_and_wait(
    name="QuickstartMemorySdk",
    description="Getting-started memory resource (SDK)",
    event_expiry_days=30,
    memory_execution_role_arn=MEMORY_ROLE_ARN,
)
memory_id = memory["id"]
print("Memory:", memory_id, memory["status"])

## 2. Write a short-term event

`create_event` accepts a list of `(text, role)` tuples — the client handles payload shaping.

In [ ]:
client.create_event(
    memory_id=memory_id,
    actor_id=ACTOR_ID,
    session_id=SESSION_ID,
    messages=[
        ("My name is Alex and I prefer Python.", "USER"),
        ("Nice to meet you, Alex.", "ASSISTANT"),
    ],
)

## 3. Read short-term history

`get_last_k_turns` returns the most recent conversation turns for the session.

In [ ]:
turns = client.get_last_k_turns(
    memory_id=memory_id, actor_id=ACTOR_ID, session_id=SESSION_ID, k=5
)
for turn in turns:
    for msg in turn:
        print(msg["role"], "→", msg["content"]["text"])

## 4. Add a built-in semantic strategy

In [ ]:
client.update_memory_strategies(
    memory_id=memory_id,
    add_strategies=[
        {
            "semanticMemoryStrategy": {
                "name": "UserFacts",
                "namespaces": ["/users/{actorId}/facts"],
            }
        }
    ],
)

# Extraction is asynchronous — give it ~60s before retrieving.
time.sleep(60)

## 5. Retrieve a memory record

In [ ]:
hits = client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"/users/{ACTOR_ID}/facts",
    query="What programming language does the user prefer?",
    top_k=3,
)
for h in hits:
    print(h["content"]["text"])

## 6. Teardown

In [ ]:
client.delete_memory_and_wait(memory_id=memory_id)

## See also

- [Concepts](./01-memory-concepts.md)
- Same flow in [CLI](./03-quickstart-cli.md) and [boto3](./04-quickstart-boto3.ipynb).